# semgrep — Multi-Language Pattern-Based Static Analysis
Install: `pip install semgrep` | CLI: `semgrep --config auto --json <path>`

In [1]:
import semgrep
print(dir(semgrep))

['__VERSION__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__']


In [2]:
import pkgutil, importlib
for finder, name, ispkg in pkgutil.walk_packages(semgrep.__path__, semgrep.__name__ + '.', onerror=lambda x: None):
    try:
        mod = importlib.import_module(name)
        print(name, '->', dir(mod))
    except Exception as e:
        print(name, '-> SKIP:', e)

semgrep.__main__ -> ['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'sys']
semgrep.app -> ['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__']


semgrep.app.auth -> ['DeploymentConfig', 'Optional', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', '_read_token_from_settings_file', 'get_deployment_from_token', 'get_deployment_id', 'get_path', 'get_state', 'get_token', 'is_a_tty', 'is_logged_in_weak', 'logger', 'logging', 'set_token', 'sys', 'telemetry']
semgrep.app.project_config -> ['Any', 'CONFIG_FILE_PATTERN', 'Dict', 'List', 'Optional', 'Path', 'ProjectConfig', 'SemgrepError', 'UNPARSEABLE_YAML_EXIT_CODE', 'YAMLError', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', '_indent', 'asdict', 'define', 'field', 'getLogger', 'get_git_root_path', 'logger', 'out', 're', 'ruamel']
semgrep.app.scans -> ['ALL_PRODUCTS', 'Counter', 'DependencyParserError', 'Dict', 'FilteredMatches', 'FrozenSet', 'INVALID_API_KEY_EXIT_CODE', 'List', 'Optional', 'ParsingData', 'Path', 'ProjectConfig', 'Rule', 'RuleMatch', 'ScanHandler', 'Set', 

semgrep.cli -> ['DefaultGroup', 'Dict', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'ci', 'cli', 'click', 'getLogger', 'get_state', 'git_check_output', 'install_semgrep_pro', 'logger', 'login', 'maybe_set_git_safe_directories', 'publish', 'scan', 'semgrep_mcp']
semgrep.commands -> ['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'ci', 'install', 'login', 'mcp', 'publish', 'scan', 'wrapper']
semgrep.commands.ci -> ['ALL_PRODUCTS', 'AutofixBehavior', 'EngineType', 'FATAL_EXIT_CODE', 'FilteredMatches', 'GitMeta', 'GithubMeta', 'INVALID_API_KEY_EXIT_CODE', 'List', 'MISSING_CONFIG_EXIT_CODE', 'Mapping', 'MemoryPolicy', 'MetricsState', 'Optional', 'OutputFormat', 'OutputHandler', 'OutputSettings', 'PRODUCT_NAMES_MAP', 'Padding', 'ParsingData', 'Path', 'Progress', 'ProjectConfig', 'Rule', 'RuleMatch', 'RuleMatchMap', 'SAST_PRODUCT', 'ScanHandler', 'SemgrepCoreErr

In [3]:
# Raw JSON output from semgrep with python security ruleset
import subprocess, os

rw = os.path.join(os.path.dirname(os.getcwd()), 'redditwarp', 'redditwarp')
r = subprocess.run(
    ['semgrep', '--config', 'p/python', '--json', '--metrics=off',
     '--timeout', '30', '--jobs', '4', rw],
    capture_output=True, text=True, encoding='utf-8', errors='replace', timeout=300
)
print(r.stdout)

{"version":"1.156.0","results":[{"check_id":"python.lang.security.insecure-uuid-version.insecure-uuid-version","path":"F:\\testable-whitebox-metrics\\redditwarp\\redditwarp\\util\\redditwarp_installed_client_credentials.py","start":{"line":18,"col":26,"offset":317},"end":{"line":18,"col":38,"offset":329},"extra":{"message":"Using UUID version 1 for UUID generation can lead to predictable UUIDs based on system information (e.g., MAC address, timestamp). This may lead to security risks such as the sandwich attack. Consider using `uuid.uuid4()` instead for better randomness and security.","fix":"uuid.uuid4()","metadata":{"references":["https://www.landh.tech/blog/20230811-sandwich-attack/"],"cwe":["CWE-330: Use of Insufficiently Random Values"],"owasp":["A02:2021 - Cryptographic Failures","A04:2025 - Cryptographic Failures"],"asvs":{"control_id":"6.3.2 Insecure UUID Generation","control_url":"https://github.com/OWASP/ASVS/blob/master/4.0/en/0x14-V6-Cryptography.md#v63-random-values","sect

In [4]:
# Raw semgrep --dump-ast output on a single file
import subprocess, os

target = os.path.join(os.path.dirname(os.getcwd()), 'redditwarp', 'redditwarp', 'spaces', 'discrete.py')
r = subprocess.run(
    ['semgrep', '--dump-ast', '--json', '--lang', 'python', target],
    capture_output=True, text=True, encoding='utf-8', errors='replace', timeout=60
)
print(r.stdout)

In [5]:
# Raw semgrep with custom rules — full JSON
import subprocess, os, tempfile

rules = '''
rules:
  - id: assert-used
    pattern: assert $X
    message: assert statement found
    languages: [python]
    severity: WARNING
  - id: bare-except
    pattern: |
      try:
          ...
      except:
          ...
    message: bare except clause
    languages: [python]
    severity: ERROR
  - id: print-call
    pattern: print($X)
    message: print() call found
    languages: [python]
    severity: INFO
'''

with tempfile.NamedTemporaryFile(mode='w', suffix='.yaml', delete=False) as f:
    f.write(rules)
    rule_file = f.name

rw = os.path.join(os.path.dirname(os.getcwd()), 'redditwarp', 'redditwarp')
r = subprocess.run(
    ['semgrep', '--config', rule_file, '--json', '--metrics=off', rw],
    capture_output=True, text=True, encoding='utf-8', errors='replace', timeout=120
)
print(r.stdout)
os.unlink(rule_file)

{"version":"1.156.0","results":[{"check_id":"C.Users.vinod.AppData.Local.Temp.print-call","path":"F:\\testable-whitebox-metrics\\redditwarp\\redditwarp\\_cli\\comment_tree.py","start":{"line":220,"col":1,"offset":7817},"end":{"line":225,"col":5,"offset":8015},"extra":{"message":"print() call found","metadata":{},"severity":"INFO","fingerprint":"requires login","lines":"requires login","validation_state":"NO_VALIDATOR","engine_kind":"OSS"}},{"check_id":"C.Users.vinod.AppData.Local.Temp.print-call","path":"F:\\testable-whitebox-metrics\\redditwarp\\redditwarp\\_cli\\comment_tree.py","start":{"line":233,"col":5,"offset":8230},"end":{"line":233,"col":26,"offset":8251},"extra":{"message":"print() call found","metadata":{},"severity":"INFO","fingerprint":"requires login","lines":"requires login","validation_state":"NO_VALIDATOR","engine_kind":"OSS"}},{"check_id":"C.Users.vinod.AppData.Local.Temp.print-call","path":"F:\\testable-whitebox-metrics\\redditwarp\\redditwarp\\_cli\\exchange_authori